# 11.16 — Generalized Advantage Estimation

Generalized Advantage Estimation (GAE) is the bridge between raw reward streams and stable policy-gradient updates: it estimates how much better an action was than the value function expected by exponentially weighting temporal-difference residuals. In this lesson, you will build the return, TD error, GAE(λ) recursion, and the bias-variance tradeoff from scratch with NumPy, then inspect how λ changes the signal a policy optimizer would receive.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build generalized advantage estimation one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the recursion is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized arithmetic, and reproducible simulation.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic trajectories.

### 1. A trajectory is a reward ledger, not one label

Reinforcement learning data arrives as a sequence: state, action, reward, next state, and eventually a terminal flag. The supervised-learning target is not sitting beside the action; it is distributed across later rewards. We start with a tiny five-step rollout and compute discounted consequences so each reward is worth less the farther it sits from the decision that caused it.

In [ ]:
rewards_w = np.array([0.0, 0.0, 1.0, 0.0, 2.0])  # delayed payoff: most reward arrives late.
gamma_w = 0.9  # discount future rewards by 10% per step.
print("rewards:", rewards_w)
print("gamma:", gamma_w)

▶ What you'll see: a trajectory where the useful evidence arrives at steps 2 and 4, not immediately.

In [ ]:
returns_w = np.zeros_like(rewards_w)  # G_t will store discounted reward from t onward.
running_w = 0.0  # accumulator for the backward recursion.
for t_w in range(len(rewards_w) - 1, -1, -1):  # walk backward so the future is already known.
    running_w = rewards_w[t_w] + gamma_w * running_w  # G_t = r_t + γ G_{t+1}.
    returns_w[t_w] = running_w  # save this step's consequence.
print("discounted returns:", np.round(returns_w, 3))
assert np.allclose(np.round(returns_w, 3), [2.122, 2.358, 2.62, 1.8, 2.0])

▶ What you'll see: early zero-reward steps still receive credit because they led to later payoff.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(rewards_w, marker="o", label="immediate reward")
plt.plot(returns_w, marker="s", label="discounted return")
plt.title("1: return spreads future reward backward")
plt.xlabel("time step")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: the return curve is high before the delayed reward because consequence is assigned backward through time.

*Why it's done this way:* the return $G_t=r_t+\gamma r_{t+1}+\gamma^2r_{t+2}+\dots$ is the full consequence of acting at time $t$. Discounting keeps far-future rewards meaningful but not free: it encodes the mathematical assumption that a reward two steps away should count as $\gamma^2$ units today.

### 2. A value baseline turns consequence into advantage

A policy gradient does not only need "was the return large?" It needs "was this action better than what the value function expected from this state?" That difference is the advantage. Subtracting a baseline does not change which actions are good on average, but it reduces variance because common state difficulty is removed before the policy update.

In [ ]:
values_w = np.array([2.5, 2.8, 3.0, 1.4, 0.6])  # value estimates before seeing the rollout.
adv_mc_w = returns_w - values_w  # Monte Carlo advantage: full return minus baseline.
print("values:", values_w)
print("MC advantages:", np.round(adv_mc_w, 3))
assert np.allclose(np.round(adv_mc_w, 3), [-0.378, -0.442, -0.38, 0.4, 1.4])

▶ What you'll see: every step gets a signed "better than expected" number.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(adv_mc_w)), adv_mc_w, color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("2: advantage = return − value baseline")
plt.xlabel("time step")
plt.ylabel("advantage")
plt.show()

▶ What you'll see: positive bars mark actions whose later outcome beat the baseline.

*Why it's done this way:* the value function $V(s_t)$ estimates expected consequence before the sampled action's randomness is known. The advantage $A_t=G_t-V(s_t)$ isolates surprise relative to that state, so a policy update increases probability for actions with positive surprise and decreases it for negative surprise.

### 3. TD residuals bootstrap one step at a time

Monte Carlo advantage waits for the full return, which can be noisy. A temporal-difference residual asks a smaller question: did the immediate reward plus the next state's value exceed the current value? This one-step surprise is

$$\delta_t=r_t+\gamma V(s_{t+1})-V(s_t).$$

At the last step, there is no next state value after termination, so the bootstrap term is zero.

In [ ]:
next_values_w = np.append(values_w[1:], 0.0)  # terminal bootstrap is zero after the final step.
deltas_w = rewards_w + gamma_w * next_values_w - values_w  # one-step TD residuals.
print("next values:", next_values_w)
print("TD residuals δ:", np.round(deltas_w, 3))
assert np.allclose(np.round(deltas_w, 3), [0.02, -0.1, -0.74, -0.86, 1.4])

▶ What you'll see: the last step has a large positive residual, while earlier steps can be positive or negative depending on the bootstrap.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(deltas_w)), deltas_w, color="darkorange")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("3: one-step TD residuals")
plt.xlabel("time step")
plt.ylabel("δ_t")
plt.show()

▶ What you'll see: TD residuals are local surprises, not full returns; they can oscillate across the rollout.

*Why it's done this way:* bootstrapping trades variance for bias. The target $r_t+\gamma V(s_{t+1})$ uses one real reward and one learned estimate, so it reacts quickly and has lower sample variance than a long return, but it inherits any error in the value function.

### 4. GAE(λ) is an exponentially weighted sum of TD residuals

GAE combines the low-variance one-step residual with the longer-horizon return by summing future residuals with weights $(\gamma\lambda)^l$:

$$\hat A_t^{\text{GAE}(\gamma,\lambda)}=\sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l}.$$

In a finite rollout we compute that sum backward with one line of recursion.

In [ ]:
lam_w = 0.8  # λ controls how far future TD residuals are allowed to flow backward.
gae_w = np.zeros_like(deltas_w)  # store A_t^GAE.
carry_w = 0.0  # this is the weighted future residual sum.
for t_w in range(len(deltas_w) - 1, -1, -1):
    carry_w = deltas_w[t_w] + gamma_w * lam_w * carry_w  # A_t = δ_t + γλ A_{t+1}.
    gae_w[t_w] = carry_w
print("GAE(lambda=0.8):", np.round(gae_w, 3))
assert np.allclose(np.round(gae_w, 3), [-0.38, -0.556, -0.633, 0.148, 1.4])

▶ What you'll see: the final positive surprise flows backward, but it is damped each step by γλ.

In [ ]:
manual0_w = deltas_w[0] + (gamma_w * lam_w) * deltas_w[1] + (gamma_w * lam_w) ** 2 * deltas_w[2] + (gamma_w * lam_w) ** 3 * deltas_w[3] + (gamma_w * lam_w) ** 4 * deltas_w[4]
print("manual A_0:", round(float(manual0_w), 3), "recursive A_0:", round(float(gae_w[0]), 3))
assert round(float(manual0_w), 3) == round(float(gae_w[0]), 3) == -0.38

▶ What you'll see: the explicit weighted sum and the backward recursion produce the same first advantage.

*Why it's done this way:* the recursion is not a trick; it is just the geometric series written efficiently. Since $A_t=\delta_t+\gamma\lambda\delta_{t+1}+(\gamma\lambda)^2\delta_{t+2}+\dots$, the tail after $\delta_t$ is exactly $\gamma\lambda A_{t+1}$. That is why one backward pass computes every advantage.

### 5. λ is the bias-variance dial

When $\lambda=0$, GAE keeps only the one-step TD residual: low variance, but biased if the value function is wrong. When $\lambda=1$, the residuals telescope into the Monte Carlo return minus value baseline: lower bias, but higher variance because every future reward enters. Values between 0 and 1 interpolate smoothly.

In [ ]:
def gae_from_deltas_w(deltas, gamma, lam):
    out = np.zeros_like(deltas)
    carry = 0.0
    for t in range(len(deltas) - 1, -1, -1):
        carry = deltas[t] + gamma * lam * carry
        out[t] = carry
    return out

lams_w = np.array([0.0, 0.5, 0.8, 1.0])
gae_grid_w = np.vstack([gae_from_deltas_w(deltas_w, gamma_w, lam) for lam in lams_w])
print("rows are λ=0, .5, .8, 1:\n", np.round(gae_grid_w, 3))
assert np.allclose(np.round(gae_grid_w[-1], 3), np.round(adv_mc_w, 3))

▶ What you'll see: λ=0 matches TD residuals, while λ=1 matches the Monte Carlo advantage.

In [ ]:
plt.figure(figsize=(5.4, 3.2))
for row_w, lam_val_w in zip(gae_grid_w, lams_w):
    plt.plot(row_w, marker="o", label=f"λ={lam_val_w:g}")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("5: λ changes how far residuals travel")
plt.xlabel("time step")
plt.ylabel("advantage estimate")
plt.legend()
plt.show()

▶ What you'll see: larger λ makes earlier steps absorb more delayed reward and look closer to full-return advantages.

*Why it's done this way:* $\lambda$ controls how much we trust bootstrapping. Small $\lambda$ trusts $V(s_{t+1})$ quickly, so estimates are steadier but biased by the critic. Large $\lambda$ waits for real sampled rewards, so estimates are less biased by the critic but more sensitive to trajectory noise.

### 6. Episode boundaries stop advantage leakage

Rollouts are often batched, and a batch may contain multiple episodes. GAE must reset at terminal states; otherwise a reward from the next episode leaks backward into the previous one. The recursion therefore multiplies the carry by a nonterminal mask.

In [ ]:
rewards2_w = np.array([1.0, 0.0, 5.0, 0.0])  # two short episodes packed together.
values2_w = np.array([0.5, 0.4, 1.0, 0.2])
next_values2_w = np.array([0.4, 0.0, 0.2, 0.0])  # bootstrap is zero at terminals.
dones2_w = np.array([0, 1, 0, 1])  # steps 1 and 3 end episodes.
nonterminal2_w = 1 - dones2_w
print("nonterminal mask:", nonterminal2_w)

▶ What you'll see: a 0 mask exactly where an episode ends.

In [ ]:
deltas2_w = rewards2_w + gamma_w * next_values2_w * nonterminal2_w - values2_w
adv2_w = np.zeros_like(deltas2_w)
carry2_w = 0.0
for t_w in range(len(deltas2_w) - 1, -1, -1):
    carry2_w = deltas2_w[t_w] + gamma_w * lam_w * nonterminal2_w[t_w] * carry2_w
    adv2_w[t_w] = carry2_w
print("deltas:", np.round(deltas2_w, 3))
print("masked GAE:", np.round(adv2_w, 3))
assert np.allclose(np.round(adv2_w, 3), [0.572, -0.4, 4.036, -0.2])

▶ What you'll see: step 1 does not borrow the large reward from step 2 because the episode boundary cuts the recursion.

In [ ]:
bad_adv2_w = np.zeros_like(deltas2_w)
bad_carry2_w = 0.0
for t_w in range(len(deltas2_w) - 1, -1, -1):
    bad_carry2_w = deltas2_w[t_w] + gamma_w * lam_w * bad_carry2_w  # intentionally missing the terminal mask.
    bad_adv2_w[t_w] = bad_carry2_w
plt.figure(figsize=(5, 3))
plt.plot(adv2_w, marker="o", label="masked correctly")
plt.plot(bad_adv2_w, marker="s", label="leaks across episode")
plt.title("6: terminal masks stop leakage")
plt.xlabel("packed time step")
plt.ylabel("advantage")
plt.legend()
plt.show()

▶ What you'll see: the unmasked curve incorrectly moves information from episode 2 into episode 1.

*Why it's done this way:* an episode boundary means the Markov chain restarted. Mathematically, $V(s_{t+1})$ and $A_{t+1}$ are not consequences of the previous action after `done=True`, so the bootstrap and the GAE carry must be multiplied by zero.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small
> numbers, prints every intermediate value with an inline `# ->`, draws one picture, and ends
> with an `assert`. Run them top to bottom.

### ✍️ Toy 1 · Discounted returns flow backward

A return is a backward ledger: start at the end of the rollout, carry future reward backward, and discount it each step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_seed = 0
t1_rng = np.random.default_rng(t1_seed)
print("rng seed:", t1_seed)  # -> 0
t1_rewards = np.array([0.0, 2.0, 0.0, 1.0])
print("rewards:", t1_rewards.tolist())  # -> [0.0, 2.0, 0.0, 1.0]
t1_gamma = 0.5
print("gamma:", t1_gamma)  # -> 0.5
t1_returns = np.zeros_like(t1_rewards)
print("initial returns:", t1_returns.tolist())  # -> [0.0, 0.0, 0.0, 0.0]
t1_running = 0.0
print("initial running:", t1_running)  # -> 0.0
for t1_t in range(len(t1_rewards) - 1, -1, -1):
    t1_running = t1_rewards[t1_t] + t1_gamma * t1_running
    t1_returns[t1_t] = t1_running
    print("running after t=", t1_t, ":", round(float(t1_running), 3))  # -> 1.0, 0.5, 2.25, 1.125
print("discounted returns:", np.round(t1_returns, 3).tolist())  # -> [1.125, 2.25, 0.5, 1.0]

plt.figure(figsize=(4.8, 3.0))
plt.plot(t1_rewards, marker="o", label="reward")
plt.plot(t1_returns, marker="s", label="return")
plt.title("Toy 1 · returns carry reward backward")
plt.xlabel("time step")
plt.ylabel("value")
plt.legend()
plt.show()
assert np.allclose(t1_returns, [1.125, 2.25, 0.5, 1.0])

▶ What you'll see: the zero-reward first step still gets return `1.125` because later rewards flow backward.

### ✍️ Toy 2 · A value baseline makes advantages signed

Subtracting a baseline converts raw consequence into "better or worse than expected" for each time step.

In [ ]:
import numpy as np

t2_seed = 0
t2_rng = np.random.default_rng(t2_seed)
print("rng seed:", t2_seed)  # -> 0
t2_returns = np.array([1.125, 2.25, 0.5, 1.0])
print("returns:", t2_returns.tolist())  # -> [1.125, 2.25, 0.5, 1.0]
t2_values = np.array([1.0, 1.5, 0.7, 1.2])
print("values:", t2_values.tolist())  # -> [1.0, 1.5, 0.7, 1.2]
t2_advantages = t2_returns - t2_values
print("advantages:", np.round(t2_advantages, 3).tolist())  # -> [0.125, 0.75, -0.2, -0.2]
t2_positive = t2_advantages > 0
print("positive flags:", t2_positive.astype(int).tolist())  # -> [1, 1, 0, 0]

plt.figure(figsize=(4.8, 3.0))
plt.bar(np.arange(t2_advantages.size), t2_advantages, color=["seagreen" if v > 0 else "indianred" for v in t2_advantages])
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 2 · advantage = return − value")
plt.xlabel("time step")
plt.ylabel("advantage")
plt.show()
assert np.allclose(np.round(t2_advantages, 3), [0.125, 0.75, -0.2, -0.2])

▶ What you'll see: the first two actions beat the baseline, while the last two fall below it.

### ✍️ Toy 3 · TD residuals bootstrap from the next value

A TD residual checks whether one reward plus the next value estimate was higher than the current value estimate.

In [ ]:
import numpy as np

t3_seed = 0
t3_rng = np.random.default_rng(t3_seed)
print("rng seed:", t3_seed)  # -> 0
t3_rewards = np.array([1.0, 0.0, 2.0])
print("rewards:", t3_rewards.tolist())  # -> [1.0, 0.0, 2.0]
t3_values = np.array([0.4, 0.8, 1.5])
print("values:", t3_values.tolist())  # -> [0.4, 0.8, 1.5]
t3_gamma = 0.9
print("gamma:", t3_gamma)  # -> 0.9
t3_next_values = np.append(t3_values[1:], 0.0)
print("next values:", t3_next_values.tolist())  # -> [0.8, 1.5, 0.0]
t3_bootstrap = t3_gamma * t3_next_values
print("discounted next values:", np.round(t3_bootstrap, 3).tolist())  # -> [0.72, 1.35, 0.0]
t3_deltas = t3_rewards + t3_bootstrap - t3_values
print("TD residuals:", np.round(t3_deltas, 3).tolist())  # -> [1.32, 0.55, 0.5]

plt.figure(figsize=(4.8, 3.0))
plt.bar(np.arange(t3_deltas.size), t3_deltas, color="darkorange")
plt.title("Toy 3 · one-step TD surprises")
plt.xlabel("time step")
plt.ylabel("δ")
plt.show()
assert np.allclose(np.round(t3_deltas, 3), [1.32, 0.55, 0.5])

▶ What you'll see: even the zero-reward middle step has positive residual because its next value is high.

### ✍️ Toy 4 · GAE recursively sums damped residuals

GAE is the same weighted residual sum written backward, so the future carry is already available.

In [ ]:
import numpy as np

t4_seed = 0
t4_rng = np.random.default_rng(t4_seed)
print("rng seed:", t4_seed)  # -> 0
t4_deltas = np.array([0.4, -0.2, 1.0])
print("TD residuals:", t4_deltas.tolist())  # -> [0.4, -0.2, 1.0]
t4_gamma = 0.9
print("gamma:", t4_gamma)  # -> 0.9
t4_lambda = 0.5
print("lambda:", t4_lambda)  # -> 0.5
t4_decay = t4_gamma * t4_lambda
print("gamma * lambda:", t4_decay)  # -> 0.45
t4_gae = np.zeros_like(t4_deltas)
print("initial GAE:", t4_gae.tolist())  # -> [0.0, 0.0, 0.0]
t4_carry = 0.0
print("initial carry:", t4_carry)  # -> 0.0
for t4_t in range(len(t4_deltas) - 1, -1, -1):
    t4_carry = t4_deltas[t4_t] + t4_decay * t4_carry
    t4_gae[t4_t] = t4_carry
    print("carry after t=", t4_t, ":", round(float(t4_carry), 4))  # -> 1.0, 0.25, 0.5125
t4_manual = t4_deltas[0] + t4_decay * t4_deltas[1] + t4_decay ** 2 * t4_deltas[2]
print("manual A0:", round(float(t4_manual), 4))  # -> 0.5125
print("recursive GAE:", np.round(t4_gae, 4).tolist())  # -> [0.5125, 0.25, 1.0]

plt.figure(figsize=(4.8, 3.0))
plt.plot(t4_deltas, marker="o", label="δ")
plt.plot(t4_gae, marker="s", label="GAE")
plt.title("Toy 4 · damped residual carry")
plt.xlabel("time step")
plt.ylabel("value")
plt.legend()
plt.show()
assert round(float(t4_manual), 4) == round(float(t4_gae[0]), 4) == 0.5125

▶ What you'll see: the last residual is strongest at the end, then reaches earlier steps through a `0.45` decay.

### ✍️ Toy 5 · Lambda controls how far residuals travel

Hold the same TD residuals fixed and sweep λ; larger λ lets later residuals affect earlier actions more.

In [ ]:
import numpy as np

t5_seed = 0
t5_rng = np.random.default_rng(t5_seed)
print("rng seed:", t5_seed)  # -> 0
t5_deltas = np.array([0.4, -0.2, 1.0])
print("TD residuals:", t5_deltas.tolist())  # -> [0.4, -0.2, 1.0]
t5_gamma = 0.9
print("gamma:", t5_gamma)  # -> 0.9
t5_lambdas = np.array([0.0, 0.5, 1.0])
print("lambdas:", t5_lambdas.tolist())  # -> [0.0, 0.5, 1.0]
t5_rows = []
for t5_lambda in t5_lambdas:
    t5_out = np.zeros_like(t5_deltas)
    t5_carry = 0.0
    for t5_t in range(len(t5_deltas) - 1, -1, -1):
        t5_carry = t5_deltas[t5_t] + t5_gamma * t5_lambda * t5_carry
        t5_out[t5_t] = t5_carry
    t5_rows.append(t5_out)
    print("GAE for lambda", float(t5_lambda), ":", np.round(t5_out, 3).tolist())  # -> [0.4, -0.2, 1.0], [0.513, 0.25, 1.0], [1.03, 0.7, 1.0]
t5_grid = np.vstack(t5_rows)
print("GAE grid:", np.round(t5_grid, 3).tolist())  # -> [[0.4, -0.2, 1.0], [0.513, 0.25, 1.0], [1.03, 0.7, 1.0]]

plt.figure(figsize=(5.0, 3.0))
for t5_row, t5_lambda in zip(t5_grid, t5_lambdas):
    plt.plot(t5_row, marker="o", label=f"λ={t5_lambda:g}")
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 5 · larger λ carries farther")
plt.xlabel("time step")
plt.ylabel("advantage")
plt.legend()
plt.show()
assert t5_grid[2, 0] > t5_grid[1, 0] > t5_grid[0, 0]

▶ What you'll see: the first advantage rises from `0.4` to `1.03` as λ grows from 0 to 1.

### ✍️ Toy 6 · Terminal masks stop leakage across episodes

When a packed batch contains two episodes, the GAE carry must be multiplied by zero at `done=True`.

In [ ]:
import numpy as np

t6_seed = 0
t6_rng = np.random.default_rng(t6_seed)
print("rng seed:", t6_seed)  # -> 0
t6_rewards = np.array([0.0, 3.0, 0.0, 2.0])
print("rewards:", t6_rewards.tolist())  # -> [0.0, 3.0, 0.0, 2.0]
t6_values = np.array([0.5, 1.0, 0.4, 0.2])
print("values:", t6_values.tolist())  # -> [0.5, 1.0, 0.4, 0.2]
t6_next_values = np.array([1.0, 0.0, 0.2, 0.0])
print("next values:", t6_next_values.tolist())  # -> [1.0, 0.0, 0.2, 0.0]
t6_dones = np.array([0, 1, 0, 1])
print("done flags:", t6_dones.tolist())  # -> [0, 1, 0, 1]
t6_mask = 1 - t6_dones
print("nonterminal mask:", t6_mask.tolist())  # -> [1, 0, 1, 0]
t6_gamma = 0.9
t6_lambda = 0.8
t6_deltas = t6_rewards + t6_gamma * t6_next_values * t6_mask - t6_values
print("masked deltas:", np.round(t6_deltas, 3).tolist())  # -> [0.4, 2.0, -0.22, 1.8]
t6_masked = np.zeros_like(t6_deltas)
t6_carry = 0.0
for t6_t in range(len(t6_deltas) - 1, -1, -1):
    t6_carry = t6_deltas[t6_t] + t6_gamma * t6_lambda * t6_mask[t6_t] * t6_carry
    t6_masked[t6_t] = t6_carry
print("masked GAE:", np.round(t6_masked, 3).tolist())  # -> [1.84, 2.0, 1.076, 1.8]
t6_unmasked = np.zeros_like(t6_deltas)
t6_bad_carry = 0.0
for t6_t in range(len(t6_deltas) - 1, -1, -1):
    t6_bad_carry = t6_deltas[t6_t] + t6_gamma * t6_lambda * t6_bad_carry
    t6_unmasked[t6_t] = t6_bad_carry
print("unmasked GAE:", np.round(t6_unmasked, 3).tolist())  # -> [2.398, 2.775, 1.076, 1.8]

plt.figure(figsize=(5.0, 3.0))
plt.plot(t6_masked, marker="o", label="masked")
plt.plot(t6_unmasked, marker="s", label="unmasked")
plt.title("Toy 6 · done mask cuts the carry")
plt.xlabel("packed time step")
plt.ylabel("advantage")
plt.legend()
plt.show()
assert round(float(t6_masked[1]), 3) == 2.0 and t6_unmasked[1] > t6_masked[1]

▶ What you'll see: without the mask, the second episode's reward leaks backward into the first episode.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, recursions, simulations, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for bars, lines, histograms, and debugging plots.
np.random.seed(0)  # make all stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Discount one delayed reward

**Goal.** Compute the present value of a reward two steps away, because GAE is built on discounted consequence rather than immediate reward. We build it in 2 steps.

In [ ]:
gamma_b1 = 0.9  # choose the discount factor used throughout many examples.
reward_late_b1 = 2.0  # place a reward two time steps in the future.
discounted_b1 = (gamma_b1 ** 2) * reward_late_b1  # compute γ² times the delayed reward.
print("discounted future reward:", round(discounted_b1, 3))
assert round(discounted_b1, 3) == 1.62

▶ What you'll see: a future reward of 2 counts as 1.62 today when γ=0.9.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["raw future", "discounted now"], [reward_late_b1, discounted_b1], color=["gray", "teal"])
plt.title("Basic 1: discounting delayed reward")
plt.ylabel("value")
plt.show()

▶ What you'll see: discounting shrinks the delayed reward without erasing it.

👀 Takeaway: discounting is the first step in assigning future consequences to current decisions.

### Basic 2 — Compute a short discounted return

**Goal.** Sum a three-step return by hand in code, because full returns are the high-λ endpoint of GAE. We build it in 2 steps.

In [ ]:
rewards_b2 = np.array([1.0, 0.0, 2.0])  # define a tiny reward sequence.
gamma_b2 = 0.9  # reuse the same discount factor.
powers_b2 = gamma_b2 ** np.arange(len(rewards_b2))  # compute [1, γ, γ²].
print("discount powers:", np.round(powers_b2, 3))

▶ What you'll see: each later reward receives one more power of γ.

In [ ]:
return_b2 = float(np.sum(powers_b2 * rewards_b2))  # compute G0 as a weighted sum.
print("G0:", round(return_b2, 3))
assert round(return_b2, 3) == 2.62
plt.figure(figsize=(4, 3))
plt.bar(["t0", "t1", "t2"], powers_b2 * rewards_b2, color="purple")
plt.title("Basic 2: terms in a discounted return")
plt.ylabel("discounted contribution")
plt.show()

▶ What you'll see: only the first and third rewards contribute, with the third discounted by γ².

👀 Takeaway: a return is a weighted sum of rewards, with weights decreasing into the future.

### Basic 3 — Subtract a value baseline

**Goal.** Turn a return into an advantage estimate, because policy updates need better-than-expected signals rather than raw rewards. We build it in 2 steps.

In [ ]:
return_b3 = 2.62  # use the discounted return from the previous idea.
value_b3 = 2.10  # suppose the critic expected 2.10 from this state.
advantage_b3 = return_b3 - value_b3  # compute A = G - V.
print("advantage:", round(advantage_b3, 3))
assert round(advantage_b3, 3) == 0.52

▶ What you'll see: the action looks 0.52 better than the critic expected.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["return G", "value V", "advantage"], [return_b3, value_b3, advantage_b3], color=["teal", "gray", "orange"])
plt.title("Basic 3: advantage is excess return")
plt.ylabel("amount")
plt.show()

▶ What you'll see: advantage is the gap between the realized consequence and the baseline.

👀 Takeaway: the value function removes state difficulty so the policy sees a cleaner learning signal.

### Basic 4 — Build a one-step TD target

**Goal.** Bootstrap from the next value estimate, because TD residuals are the ingredients GAE exponentially weights. We build it in 2 steps.

In [ ]:
reward_b4 = 1.0  # observed immediate reward.
next_value_b4 = 0.8  # critic estimate for the next state.
gamma_b4 = 0.9  # discount factor.
target_b4 = reward_b4 + gamma_b4 * next_value_b4  # y = r + γ V(s').
print("TD target:", round(target_b4, 3))
assert round(target_b4, 3) == 1.72

▶ What you'll see: the one-step target combines real reward with one bootstrapped future estimate.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["reward", "γ next V", "target"], [reward_b4, gamma_b4 * next_value_b4, target_b4], color="seagreen")
plt.title("Basic 4: TD target pieces")
plt.ylabel("value")
plt.show()

▶ What you'll see: the target is the sum of immediate evidence and discounted bootstrap evidence.

👀 Takeaway: TD targets are cheaper and usually less noisy than waiting for the whole future return.

### Basic 5 — Compute one TD residual

**Goal.** Compare the TD target with the current value, because GAE is an exponentially weighted sequence of these residuals. We build it in 2 steps.

In [ ]:
value_b5 = 0.4  # current value estimate for the state.
target_b5 = 1.72  # use the one-step target from Basic 4.
delta_b5 = target_b5 - value_b5  # δ = r + γV(s') - V(s).
print("TD residual δ:", round(delta_b5, 3))
assert round(delta_b5, 3) == 1.32

▶ What you'll see: the critic underpredicted this one-step target by 1.32.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["target", "value", "δ"], [target_b5, value_b5, delta_b5], color=["teal", "gray", "crimson"])
plt.title("Basic 5: TD residual")
plt.ylabel("value")
plt.show()

▶ What you'll see: the residual is the vertical gap between target and current value.

👀 Takeaway: a positive TD residual means the observed transition was better than the critic predicted.

### Basic 6 — Run the GAE recursion backward

**Goal.** Implement $A_t=\delta_t+\gamma\lambda A_{t+1}$ for a tiny sequence, because this is the core GAE algorithm. We build it in 3 steps.

In [ ]:
deltas_b6 = np.array([0.5, -0.2, 1.0])  # three local TD surprises.
gamma_b6 = 0.9  # discount factor.
lam_b6 = 0.8  # trace decay parameter.
print("deltas:", deltas_b6)

▶ What you'll see: the residual sequence contains one negative local surprise and a later positive one.

In [ ]:
adv_b6 = np.zeros_like(deltas_b6)  # allocate the advantage sequence.
carry_b6 = 0.0  # future weighted residual accumulator.
for t_b6 in range(len(deltas_b6) - 1, -1, -1):
    carry_b6 = deltas_b6[t_b6] + gamma_b6 * lam_b6 * carry_b6  # backward GAE recursion.
    adv_b6[t_b6] = carry_b6
print("GAE advantages:", np.round(adv_b6, 3))
assert np.allclose(np.round(adv_b6, 3), [0.874, 0.52, 1.0])

▶ What you'll see: the final positive residual flows backward, discounted by γλ at each step.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(deltas_b6, marker="o", label="δ")
plt.plot(adv_b6, marker="s", label="GAE")
plt.title("Basic 6: backward GAE recursion")
plt.xlabel("time")
plt.legend()
plt.show()

▶ What you'll see: advantages are smoother than raw residuals because they include future residuals.

👀 Takeaway: GAE is a backward discounted filter over TD errors.

### Basic 7 — Verify recursion equals an explicit sum

**Goal.** Expand one GAE value as a geometric weighted sum, because the recursion is easier to trust when the math is visible. We build it in 2 steps.

In [ ]:
deltas_b7 = np.array([0.5, -0.2, 1.0])  # reuse the same residual pattern.
gamma_b7 = 0.9  # discount factor.
lam_b7 = 0.8  # trace decay.
weights_b7 = (gamma_b7 * lam_b7) ** np.arange(len(deltas_b7))  # [1, γλ, (γλ)^2].
print("weights:", np.round(weights_b7, 3))

▶ What you'll see: future residuals receive geometrically smaller weights.

In [ ]:
explicit_b7 = float(np.sum(weights_b7 * deltas_b7))  # A0 by the written-out sum.
print("explicit A0:", round(explicit_b7, 3))
assert round(explicit_b7, 3) == 0.874
plt.figure(figsize=(4, 3))
plt.bar(["δ0", "γλδ1", "(γλ)²δ2"], weights_b7 * deltas_b7, color="darkorange")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 7: weighted residual sum")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the negative middle residual subtracts, but the later positive residual still contributes.

👀 Takeaway: the GAE recursion is just a compact way to compute an exponentially weighted residual sum.

### Basic 8 — Compare λ = 0 and λ = 1

**Goal.** Inspect the two endpoints of λ, because they reveal the bias-variance tradeoff behind GAE. We build it in 3 steps.

In [ ]:
deltas_b8 = np.array([0.5, -0.2, 1.0])  # choose residuals with mixed signs.
gamma_b8 = 0.9  # discount factor.
print("deltas:", deltas_b8)

▶ What you'll see: the raw residuals are the λ=0 estimate.

In [ ]:
adv0_b8 = deltas_b8.copy()  # λ=0 means A_t = δ_t only.
adv1_b8 = np.zeros_like(deltas_b8)  # λ=1 uses discounted future residuals.
carry_b8 = 0.0
for t_b8 in range(len(deltas_b8) - 1, -1, -1):
    carry_b8 = deltas_b8[t_b8] + gamma_b8 * carry_b8  # γλ with λ=1.
    adv1_b8[t_b8] = carry_b8
print("λ=0:", np.round(adv0_b8, 3), "λ=1:", np.round(adv1_b8, 3))
assert np.allclose(np.round(adv1_b8, 3), [1.13, 0.7, 1.0])

▶ What you'll see: λ=1 propagates more future information backward than λ=0.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(adv0_b8, marker="o", label="λ=0")
plt.plot(adv1_b8, marker="s", label="λ=1")
plt.title("Basic 8: GAE endpoints")
plt.xlabel("time")
plt.ylabel("advantage")
plt.legend()
plt.show()

▶ What you'll see: the λ=1 curve is more influenced by the later positive residual.

👀 Takeaway: λ=0 is most bootstrapped; λ=1 is closest to full-return learning.

### Basic 9 — Stop recursion at a terminal state

**Goal.** Apply a done mask in the recursion, because advantages must not leak across separate episodes. We build it in 2 steps.

In [ ]:
deltas_b9 = np.array([0.5, -0.2, 1.0])  # residuals from two packed episodes.
dones_b9 = np.array([0, 1, 0])  # step 1 terminates an episode.
nonterminal_b9 = 1 - dones_b9  # 1 means the next step belongs to the same episode.
print("nonterminal mask:", nonterminal_b9)

▶ What you'll see: the middle step has a zero continuation mask.

In [ ]:
adv_b9 = np.zeros_like(deltas_b9)
carry_b9 = 0.0
for t_b9 in range(len(deltas_b9) - 1, -1, -1):
    carry_b9 = deltas_b9[t_b9] + 0.9 * 0.8 * nonterminal_b9[t_b9] * carry_b9
    adv_b9[t_b9] = carry_b9
print("masked advantages:", np.round(adv_b9, 3))
assert np.allclose(np.round(adv_b9, 3), [0.356, -0.2, 1.0])
plt.figure(figsize=(4, 3))
plt.bar(["t0", "t1 done", "t2"], adv_b9, color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 9: terminal mask in GAE")
plt.show()

▶ What you'll see: t1 does not borrow t2's residual because t1 ended its episode.

👀 Takeaway: terminal masks enforce the causal boundary between episodes.

### Basic 10 — Center advantages before a policy update

**Goal.** Normalize advantages to zero mean and unit scale, because policy optimizers often benefit from a stable update magnitude. We build it in 3 steps.

In [ ]:
adv_b10 = np.array([0.874, 0.52, 1.0, -0.2, 0.1])  # a small batch of advantage estimates.
mean_b10 = float(np.mean(adv_b10))  # compute the batch center.
std_b10 = float(np.std(adv_b10))  # compute the batch scale.
print("mean:", round(mean_b10, 3), "std:", round(std_b10, 3))

▶ What you'll see: the raw batch has a positive mean and a non-unit scale.

In [ ]:
norm_adv_b10 = (adv_b10 - mean_b10) / (std_b10 + 1e-8)  # normalize with a tiny guard against divide-by-zero.
print("normalized advantages:", np.round(norm_adv_b10, 3))
assert abs(float(np.mean(norm_adv_b10))) < 1e-8
assert round(float(np.std(norm_adv_b10)), 3) == 1.0

▶ What you'll see: normalized advantages are centered around zero with standard deviation one.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(len(adv_b10)) - 0.15, adv_b10, width=0.3, label="raw")
plt.bar(np.arange(len(norm_adv_b10)) + 0.15, norm_adv_b10, width=0.3, label="normalized")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 10: advantage normalization")
plt.legend()
plt.show()

▶ What you'll see: normalization preserves which samples are above or below average while changing scale.

👀 Takeaway: advantage normalization changes update scale, not the ordering of good versus bad sampled actions.

## 🟡 Easy

### Easy 1 — Compute GAE for a whole rollout

**Goal.** Combine rewards, values, dones, γ, and λ into one reusable NumPy implementation, because every policy-gradient batch needs this calculation. We build it in 4 steps.

In [ ]:
rewards_e1 = np.array([0.0, 0.0, 1.0, 0.0, 2.0])  # define the rollout rewards.
values_e1 = np.array([2.5, 2.8, 3.0, 1.4, 0.6])  # define critic predictions for current states.
dones_e1 = np.array([0, 0, 0, 0, 1])  # mark only the final step as terminal.
gamma_e1 = 0.9
lam_e1 = 0.8
print("rollout length:", len(rewards_e1))

▶ What you'll see: one five-step episode ready for advantage estimation.

In [ ]:
next_values_e1 = np.append(values_e1[1:], 0.0)  # final next value is zero at terminal.
nonterminal_e1 = 1 - dones_e1  # 1 where bootstrapping is allowed.
deltas_e1 = rewards_e1 + gamma_e1 * next_values_e1 * nonterminal_e1 - values_e1  # TD residuals.
print("deltas:", np.round(deltas_e1, 3))
assert np.allclose(np.round(deltas_e1, 3), [0.02, -0.1, -0.74, -0.86, 1.4])

▶ What you'll see: the same local surprises computed in the walkthrough.

In [ ]:
adv_e1 = np.zeros_like(rewards_e1)
carry_e1 = 0.0
for t_e1 in range(len(rewards_e1) - 1, -1, -1):
    carry_e1 = deltas_e1[t_e1] + gamma_e1 * lam_e1 * nonterminal_e1[t_e1] * carry_e1
    adv_e1[t_e1] = carry_e1
print("GAE:", np.round(adv_e1, 3))
assert np.allclose(np.round(adv_e1, 3), [-0.38, -0.556, -0.633, 0.148, 1.4])

▶ What you'll see: GAE advantages for every time step in one backward pass.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(deltas_e1, marker="o", label="TD residual δ")
plt.plot(adv_e1, marker="s", label="GAE λ=.8")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 1: GAE from a rollout")
plt.xlabel("time")
plt.legend()
plt.show()

▶ What you'll see: the GAE curve smooths and propagates TD residuals backward.

👀 Takeaway: GAE is computed by first forming TD residuals, then running one masked backward recursion.

### Easy 2 — Visualize the λ effect

**Goal.** Sweep λ values for the same residuals, because λ is the visible bias-variance knob in GAE. We build it in 3 steps.

In [ ]:
deltas_e2 = np.array([0.02, -0.1, -0.74, -0.86, 1.4])  # reuse the rollout's TD residuals.
gamma_e2 = 0.9
lams_e2 = np.array([0.0, 0.3, 0.6, 0.9, 1.0])  # compare short and long traces.
print("lambda grid:", lams_e2)

▶ What you'll see: five trace lengths from pure one-step TD to full Monte Carlo-style propagation.

In [ ]:
curves_e2 = []
for lam_e2_val in lams_e2:
    out_e2 = np.zeros_like(deltas_e2)
    carry_e2 = 0.0
    for t_e2 in range(len(deltas_e2) - 1, -1, -1):
        carry_e2 = deltas_e2[t_e2] + gamma_e2 * lam_e2_val * carry_e2
        out_e2[t_e2] = carry_e2
    curves_e2.append(out_e2)
curves_e2 = np.array(curves_e2)
print("A0 by λ:", np.round(curves_e2[:, 0], 3))
assert np.allclose(np.round(curves_e2[:, 0], 3), [0.02, -0.07, -0.266, -0.401, -0.378])

▶ What you'll see: the first-step advantage changes as more future residuals are admitted.

In [ ]:
plt.figure(figsize=(5.5, 3.2))
for curve_e2, lam_e2_val in zip(curves_e2, lams_e2):
    plt.plot(curve_e2, marker="o", label=f"λ={lam_e2_val:g}")
plt.title("Easy 2: λ controls trace length")
plt.xlabel("time")
plt.ylabel("advantage")
plt.legend(ncol=2)
plt.show()

▶ What you'll see: larger λ curves are more affected by late rewards; look especially at the leftmost point.

👀 Takeaway: increasing λ lets delayed evidence travel farther backward, usually reducing bias and increasing variance.

### Easy 3 — Compare GAE returns to value targets

**Goal.** Convert advantages back into critic targets with $R_t^{\lambda}=A_t+V(s_t)$, because actor-critic training often updates the value function toward these λ-returns. We build it in 3 steps.

In [ ]:
values_e3 = np.array([2.5, 2.8, 3.0, 1.4, 0.6])  # current critic predictions.
adv_e3 = np.array([-0.38, -0.556, -0.633, 0.148, 1.4])  # GAE advantages from λ=.8.
lambda_returns_e3 = adv_e3 + values_e3  # target for the critic.
print("lambda returns:", np.round(lambda_returns_e3, 3))
assert np.allclose(np.round(lambda_returns_e3, 3), [2.12, 2.244, 2.367, 1.548, 2.0])

▶ What you'll see: adding the baseline back turns advantages into value targets.

In [ ]:
mc_returns_e3 = np.array([2.122, 2.358, 2.62, 1.8, 2.0])  # full Monte Carlo returns for comparison.
print("MC returns:", mc_returns_e3)
print("λ-return minus MC:", np.round(lambda_returns_e3 - mc_returns_e3, 3))

▶ What you'll see: λ-returns sit between pure bootstrapping and full Monte Carlo targets.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lambda_returns_e3, marker="o", label="λ-return")
plt.plot(mc_returns_e3, marker="s", label="Monte Carlo return")
plt.title("Easy 3: critic targets from GAE")
plt.xlabel("time")
plt.ylabel("target value")
plt.legend()
plt.show()

▶ What you'll see: λ-returns can be smoother or closer to the critic than full returns, depending on λ and value errors.

👀 Takeaway: GAE advantages update the policy, and advantage plus value gives a λ-return for critic training.

### Easy 4 — Handle two episodes in one batch

**Goal.** Compute GAE on a packed batch with terminal masks, because real rollouts often concatenate episodes. We build it in 4 steps.

In [ ]:
rewards_e4 = np.array([1.0, 0.0, 5.0, 0.0])  # two episodes: steps 0-1 and 2-3.
values_e4 = np.array([0.5, 0.4, 1.0, 0.2])
dones_e4 = np.array([0, 1, 0, 1])
gamma_e4 = 0.9
lam_e4 = 0.8
print("dones:", dones_e4)

▶ What you'll see: two terminal points in one vector.

In [ ]:
next_values_e4 = np.array([0.4, 0.0, 0.2, 0.0])
nonterminal_e4 = 1 - dones_e4
deltas_e4 = rewards_e4 + gamma_e4 * next_values_e4 * nonterminal_e4 - values_e4
print("deltas:", np.round(deltas_e4, 3))
assert np.allclose(np.round(deltas_e4, 3), [0.86, -0.4, 4.18, -0.2])

▶ What you'll see: terminal steps have no bootstrapped next value.

In [ ]:
adv_e4 = np.zeros_like(deltas_e4)
carry_e4 = 0.0
for t_e4 in range(len(deltas_e4) - 1, -1, -1):
    carry_e4 = deltas_e4[t_e4] + gamma_e4 * lam_e4 * nonterminal_e4[t_e4] * carry_e4
    adv_e4[t_e4] = carry_e4
print("masked GAE:", np.round(adv_e4, 3))
assert np.allclose(np.round(adv_e4, 3), [0.572, -0.4, 4.036, -0.2])

▶ What you'll see: each episode gets its own independent backward trace.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(adv_e4)), adv_e4, color="purple")
for idx_e4, done_e4 in enumerate(dones_e4):
    if done_e4:
        plt.axvline(idx_e4 + 0.5, color="red", linestyle="--")
plt.title("Easy 4: GAE resets after terminal steps")
plt.xlabel("packed step")
plt.ylabel("advantage")
plt.show()

▶ What you'll see: red boundaries mark where advantage traces must stop.

👀 Takeaway: done masks are not optional bookkeeping; they preserve the meaning of consequence.

### Easy 5 — Use advantages in a policy-gradient term

**Goal.** Multiply log-probability gradients by advantages, because GAE's output becomes the weight on a policy update. We build it in 3 steps.

In [ ]:
logprob_grad_e5 = np.array([0.4, -0.1, 0.3, -0.2])  # toy ∇ log π terms for sampled actions.
adv_e5 = np.array([1.2, -0.5, 0.0, 0.8])  # toy advantages from an estimator such as GAE.
weighted_terms_e5 = logprob_grad_e5 * adv_e5  # each sample's policy-gradient contribution.
print("weighted terms:", np.round(weighted_terms_e5, 3))
assert np.allclose(np.round(weighted_terms_e5, 3), [0.48, 0.05, 0.0, -0.16])

▶ What you'll see: zero advantage contributes no policy pressure, while signs combine with the log-prob gradient.

In [ ]:
policy_grad_estimate_e5 = float(np.mean(weighted_terms_e5))  # average the sample contributions.
print("mean policy-gradient estimate:", round(policy_grad_estimate_e5, 3))
assert round(policy_grad_estimate_e5, 3) == 0.092

▶ What you'll see: the batch average is the update direction estimate in this one-parameter toy example.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(weighted_terms_e5)), weighted_terms_e5, color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 5: advantages weight policy gradients")
plt.xlabel("sample")
plt.ylabel("∇logπ × A")
plt.show()

▶ What you'll see: positive and negative samples push the policy parameter in opposite directions.

👀 Takeaway: GAE does not update the policy by itself; it supplies the advantage weights for the policy-gradient objective.

## 🔴 Advanced

### Advanced 1 — Estimate bias and variance across noisy rollouts

**Goal.** Simulate many noisy trajectories and compare λ values, because GAE's most important practical role is tuning bias versus variance. We build it in 5 steps.

In [ ]:
rng_a1 = np.random.default_rng(1)  # local randomness for reproducible rollouts.
n_rollouts_a1 = 500  # enough samples to make variance visible without a long runtime.
gamma_a1 = 0.9
lams_a1 = np.array([0.0, 0.5, 0.95, 1.0])
true_rewards_a1 = np.array([0.0, 0.0, 1.0, 0.0, 2.0])
print("rollouts:", n_rollouts_a1, "lambda grid:", lams_a1)

▶ What you'll see: one experiment will reuse the same reward pattern with Gaussian noise.

In [ ]:
true_returns_a1 = np.zeros_like(true_rewards_a1)
run_a1 = 0.0
for t_a1 in range(len(true_rewards_a1) - 1, -1, -1):
    run_a1 = true_rewards_a1[t_a1] + gamma_a1 * run_a1
    true_returns_a1[t_a1] = run_a1
biased_values_a1 = true_returns_a1 - 0.3  # critic is biased low by 0.3 at every state.
true_adv0_a1 = true_returns_a1[0] - biased_values_a1[0]
print("true A0 against biased critic:", round(float(true_adv0_a1), 3))
assert round(float(true_adv0_a1), 3) == 0.3

▶ What you'll see: if full returns were observed without noise, A0 would be 0.3 above the biased critic.

In [ ]:
estimates_a1 = {lam_a1: [] for lam_a1 in lams_a1}
for _ in range(n_rollouts_a1):
    noisy_rewards_a1 = true_rewards_a1 + rng_a1.normal(0.0, 0.6, size=true_rewards_a1.shape)  # stochastic rewards.
    next_values_a1 = np.append(biased_values_a1[1:], 0.0)
    deltas_a1 = noisy_rewards_a1 + gamma_a1 * next_values_a1 - biased_values_a1
    for lam_a1 in lams_a1:
        carry_a1 = 0.0
        for t_a1 in range(len(deltas_a1) - 1, -1, -1):
            carry_a1 = deltas_a1[t_a1] + gamma_a1 * lam_a1 * carry_a1
        estimates_a1[lam_a1].append(carry_a1)  # after the backward loop, carry is A0.
means_a1 = np.array([np.mean(estimates_a1[lam_a1]) for lam_a1 in lams_a1])
stds_a1 = np.array([np.std(estimates_a1[lam_a1]) for lam_a1 in lams_a1])
print("mean A0:", np.round(means_a1, 3))
print("std A0:", np.round(stds_a1, 3))

▶ What you'll see: larger λ tends to move the mean closer to the full-return advantage while increasing spread.

In [ ]:
bias_abs_a1 = np.abs(means_a1 - true_adv0_a1)
print("absolute bias:", np.round(bias_abs_a1, 3))
assert stds_a1[-1] > stds_a1[0]

▶ What you'll see: the Monte Carlo endpoint has the largest standard deviation in this noisy simulation.

In [ ]:
x_a1 = np.arange(len(lams_a1))
plt.figure(figsize=(5.5, 3.2))
plt.bar(x_a1 - 0.18, bias_abs_a1, width=0.36, label="|bias|", color="orange")
plt.bar(x_a1 + 0.18, stds_a1, width=0.36, label="std", color="teal")
plt.xticks(x_a1, [f"λ={v:g}" for v in lams_a1])
plt.title("Advanced 1: λ bias-variance tradeoff")
plt.ylabel("A0 error scale")
plt.legend()
plt.show()

▶ What you'll see: lower λ is steadier, higher λ is noisier; the best λ balances the two bars for the task.

👀 Takeaway: λ is a statistical tradeoff, not a magic constant; it should be chosen for the critic quality and reward noise.

### Advanced 2 — Show how value bias affects λ choices

**Goal.** Compare a good critic and a bad critic, because bootstrapping is only safe when the value function is reliable. We build it in 4 steps.

In [ ]:
rewards_a2 = np.array([0.0, 0.0, 1.0, 0.0, 2.0])
gamma_a2 = 0.9
true_returns_a2 = np.zeros_like(rewards_a2)
run_a2 = 0.0
for t_a2 in range(len(rewards_a2) - 1, -1, -1):
    run_a2 = rewards_a2[t_a2] + gamma_a2 * run_a2
    true_returns_a2[t_a2] = run_a2
print("true returns:", np.round(true_returns_a2, 3))

▶ What you'll see: the full returns provide the low-bias reference for this deterministic rollout.

In [ ]:
good_values_a2 = true_returns_a2 - 0.1  # small critic error.
bad_values_a2 = true_returns_a2 - np.array([1.0, -0.5, 0.8, -0.2, 1.5])  # uneven critic error.
lams_a2 = np.array([0.0, 0.5, 0.95, 1.0])
print("good value first:", round(float(good_values_a2[0]), 3), "bad value first:", round(float(bad_values_a2[0]), 3))

▶ What you'll see: the bad critic has larger, uneven errors that bootstrapping can inherit.

In [ ]:
rmse_good_a2 = []
rmse_bad_a2 = []
for lam_a2 in lams_a2:
    for vals_a2, store_a2 in [(good_values_a2, rmse_good_a2), (bad_values_a2, rmse_bad_a2)]:
        next_vals_a2 = np.append(vals_a2[1:], 0.0)
        deltas_a2 = rewards_a2 + gamma_a2 * next_vals_a2 - vals_a2
        adv_a2 = np.zeros_like(deltas_a2)
        carry_a2 = 0.0
        for t_a2 in range(len(deltas_a2) - 1, -1, -1):
            carry_a2 = deltas_a2[t_a2] + gamma_a2 * lam_a2 * carry_a2
            adv_a2[t_a2] = carry_a2
        target_a2 = adv_a2 + vals_a2
        store_a2.append(float(np.sqrt(np.mean((target_a2 - true_returns_a2) ** 2))))
print("RMSE good critic:", np.round(rmse_good_a2, 3))
print("RMSE bad critic:", np.round(rmse_bad_a2, 3))
assert round(rmse_good_a2[-1], 3) == 0.0 and round(rmse_bad_a2[-1], 3) == 0.0

▶ What you'll see: λ=1 recovers exact deterministic returns, while low λ depends heavily on critic quality.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lams_a2, rmse_good_a2, marker="o", label="good critic")
plt.plot(lams_a2, rmse_bad_a2, marker="s", label="bad critic")
plt.title("Advanced 2: bootstrapping inherits critic error")
plt.xlabel("λ")
plt.ylabel("λ-return RMSE vs true return")
plt.legend()
plt.show()

▶ What you'll see: the bad critic benefits more from larger λ because relying on its bootstrap is costly.

👀 Takeaway: low λ reduces sample variance only when the critic is accurate enough to trust.

### Advanced 3 — Batch GAE with variable-length episodes

**Goal.** Compute advantages for two padded episodes at once, because practical rollouts often arrive as rectangular arrays with masks. We build it in 5 steps.

In [ ]:
rewards_a3 = np.array([[1.0, 0.0, 2.0, 0.0], [0.0, 3.0, 0.0, 0.0]])  # second row is padded after step 1.
values_a3 = np.array([[0.5, 0.4, 0.6, 0.0], [0.2, 0.7, 0.0, 0.0]])
valid_a3 = np.array([[1, 1, 1, 0], [1, 1, 0, 0]])  # 1 for real steps, 0 for padding.
dones_a3 = np.array([[0, 0, 1, 0], [0, 1, 0, 0]])
print("valid mask:\n", valid_a3)

▶ What you'll see: the batch has two episodes padded to the same length.

In [ ]:
gamma_a3 = 0.9
lam_a3 = 0.8
next_values_a3 = np.concatenate([values_a3[:, 1:], np.zeros((values_a3.shape[0], 1))], axis=1)
nonterminal_a3 = (1 - dones_a3) * valid_a3
deltas_a3 = (rewards_a3 + gamma_a3 * next_values_a3 * nonterminal_a3 - values_a3) * valid_a3
print("deltas:\n", np.round(deltas_a3, 3))

▶ What you'll see: padded positions have zero residuals and will stay zero.

In [ ]:
adv_a3 = np.zeros_like(rewards_a3)
carry_a3 = np.zeros(rewards_a3.shape[0])
for t_a3 in range(rewards_a3.shape[1] - 1, -1, -1):
    carry_a3 = deltas_a3[:, t_a3] + gamma_a3 * lam_a3 * nonterminal_a3[:, t_a3] * carry_a3
    adv_a3[:, t_a3] = carry_a3 * valid_a3[:, t_a3]
print("batched advantages:\n", np.round(adv_a3, 3))
assert np.allclose(np.round(adv_a3, 3), [[1.687, 1.148, 1.4, 0.0], [2.086, 2.3, 0.0, 0.0]])

▶ What you'll see: each row receives its own recursion while padding remains exactly zero.

In [ ]:
lambda_returns_a3 = adv_a3 + values_a3 * valid_a3
print("lambda returns:\n", np.round(lambda_returns_a3, 3))

▶ What you'll see: value targets are defined only on real time steps.

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(adv_a3, cmap="coolwarm", aspect="auto")
plt.colorbar(label="advantage")
plt.title("Advanced 3: batched variable-length GAE")
plt.xlabel("time")
plt.ylabel("episode")
plt.show()

▶ What you'll see: padded cells stay neutral while real episode cells contain advantages.

👀 Takeaway: batching GAE means carrying one recursion per episode and multiplying by validity and terminal masks.

### Advanced 4 — Tune λ by mean-squared target error

**Goal.** Use a small validation-style sweep to choose λ, because the best trace length depends on reward noise and critic error. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)
gamma_a4 = 0.9
lams_a4 = np.linspace(0, 1, 6)
base_rewards_a4 = np.array([0.0, 0.0, 1.0, 0.0, 2.0])
print("lambda candidates:", np.round(lams_a4, 2))

▶ What you'll see: the sweep tests λ from 0 to 1 in even steps.

In [ ]:
true_returns_a4 = np.zeros_like(base_rewards_a4)
run_a4 = 0.0
for t_a4 in range(len(base_rewards_a4) - 1, -1, -1):
    run_a4 = base_rewards_a4[t_a4] + gamma_a4 * run_a4
    true_returns_a4[t_a4] = run_a4
values_a4 = true_returns_a4 - np.array([0.4, 0.2, -0.2, 0.1, 0.3])
print("critic values:", np.round(values_a4, 3))

▶ What you'll see: the critic is close but not perfect, so neither endpoint is guaranteed best under noise.

In [ ]:
mse_a4 = []
for lam_a4 in lams_a4:
    errs_a4 = []
    for _ in range(300):
        rewards_sample_a4 = base_rewards_a4 + rng_a4.normal(0, 0.35, size=base_rewards_a4.shape)
        next_values_a4 = np.append(values_a4[1:], 0.0)
        deltas_a4 = rewards_sample_a4 + gamma_a4 * next_values_a4 - values_a4
        adv_a4 = np.zeros_like(deltas_a4)
        carry_a4 = 0.0
        for t_a4 in range(len(deltas_a4) - 1, -1, -1):
            carry_a4 = deltas_a4[t_a4] + gamma_a4 * lam_a4 * carry_a4
            adv_a4[t_a4] = carry_a4
        target_a4 = adv_a4 + values_a4
        errs_a4.append(np.mean((target_a4 - true_returns_a4) ** 2))
    mse_a4.append(float(np.mean(errs_a4)))
best_lam_a4 = float(lams_a4[int(np.argmin(mse_a4))])
print("MSE by λ:", np.round(mse_a4, 3))
print("best λ:", best_lam_a4)
assert best_lam_a4 in lams_a4

▶ What you'll see: one λ gives the smallest average target error in this synthetic setting.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lams_a4, mse_a4, marker="o", color="purple")
plt.axvline(best_lam_a4, color="red", linestyle="--", label=f"best λ={best_lam_a4:g}")
plt.title("Advanced 4: choosing λ by validation error")
plt.xlabel("λ")
plt.ylabel("mean squared target error")
plt.legend()
plt.show()

▶ What you'll see: the curve visualizes the practical λ tradeoff; look for the lowest point, not always an endpoint.

👀 Takeaway: λ can be tuned like any other hyperparameter when you have a criterion for target quality.

### Advanced 5 — Detect advantage leakage with a diagnostic plot

**Goal.** Compare masked and unmasked GAE on a packed rollout, because a missing terminal mask creates subtle but serious credit-assignment bugs. We build it in 4 steps.

In [ ]:
deltas_a5 = np.array([0.2, -0.1, 3.0, 0.4, -0.2])  # residuals from two packed episodes.
dones_a5 = np.array([0, 1, 0, 0, 1])  # step 1 ends episode one, step 4 ends episode two.
gamma_a5 = 0.9
lam_a5 = 0.95
print("dones:", dones_a5)

▶ What you'll see: the large residual at step 2 belongs to a new episode.

In [ ]:
masked_a5 = np.zeros_like(deltas_a5)
unmasked_a5 = np.zeros_like(deltas_a5)
carry_masked_a5 = 0.0
carry_unmasked_a5 = 0.0
for t_a5 in range(len(deltas_a5) - 1, -1, -1):
    carry_masked_a5 = deltas_a5[t_a5] + gamma_a5 * lam_a5 * (1 - dones_a5[t_a5]) * carry_masked_a5
    carry_unmasked_a5 = deltas_a5[t_a5] + gamma_a5 * lam_a5 * carry_unmasked_a5
    masked_a5[t_a5] = carry_masked_a5
    unmasked_a5[t_a5] = carry_unmasked_a5
print("masked:", np.round(masked_a5, 3))
print("unmasked:", np.round(unmasked_a5, 3))
assert round(float(unmasked_a5[1] - masked_a5[1]), 3) == 2.732

▶ What you'll see: the unmasked version gives step 1 credit for rewards that happened after its episode ended.

In [ ]:
leak_a5 = unmasked_a5 - masked_a5
print("leak amount:", np.round(leak_a5, 3))

▶ What you'll see: nonzero leak values identify exactly where the bug changes the learning signal.

In [ ]:
plt.figure(figsize=(5.5, 3.2))
plt.plot(masked_a5, marker="o", label="masked GAE")
plt.plot(unmasked_a5, marker="s", label="unmasked bug")
for idx_a5, done_a5 in enumerate(dones_a5):
    if done_a5:
        plt.axvline(idx_a5 + 0.5, color="red", linestyle="--", alpha=0.7)
plt.title("Advanced 5: terminal-mask leakage diagnostic")
plt.xlabel("packed time step")
plt.ylabel("advantage")
plt.legend()
plt.show()

▶ What you'll see: the largest visual gap appears just before an episode boundary, where leakage is most dangerous.

👀 Takeaway: if masked and unmasked GAE differ across a terminal boundary, the unmasked version is assigning credit to the wrong episode.